# Who Pays Chicago's Parking Ticket Revenue Machine?

**An investigation into the city sticker fine increase and its distributional impact**

**Author:** Khushan Shahad | University of Chicago  
**Data:** ProPublica Illinois — Chicago Parking Tickets (1% sample)  
**Source:** [github.com/propublica/il-tickets-notebooks](https://github.com/propublica/il-tickets-notebooks)

---

## The Question

In 2012, the City of Chicago quietly increased the fine for missing or improperly displayed city vehicle stickers from \$120 to \$200 — a 67% increase targeting one of the most commonly issued tickets in the city.

City sticker violations are not random. They fall disproportionately on residents who cannot afford to keep up with annual sticker renewals — a cost that compounds when fines double if unpaid within a set window. This notebook investigates:

1. How did the fine increase affect the volume and revenue of city sticker tickets over time?
2. What does the revenue math look like — and who ultimately paid?
3. Are there patterns in when and where these tickets are issued?

---

## Setup

In [ ]:
import pandas as pd
import altair as alt
import numpy as np
import warnings

warnings.filterwarnings('ignore')
alt.renderers.enable('default')
alt.data_transformers.disable_max_rows()

# Update this path to wherever your data file lives
DATA_PATH = 'data/parking_tickets_one_percent.csv'

df = pd.read_csv(DATA_PATH, index_col=0)
df['issue_date'] = pd.to_datetime(df['issue_date'])
df['year'] = df['issue_date'].dt.year
df['YearMonth'] = df['issue_date'].dt.to_period('M').dt.to_timestamp()

print(f"Dataset loaded: {len(df):,} tickets")
print(f"Date range: {df['issue_date'].min().date()} to {df['issue_date'].max().date()}")
df.head(3)

---

## Part 1: Identifying City Sticker Violations

The city sticker requirement applies to all vehicles registered in Chicago. Residents must purchase an annual sticker — a flat fee regardless of income. Failure to display one results in a ticket, and unpaid tickets quickly escalate.

The relevant violation codes changed over time, which itself tells a story about how the city manages this revenue stream.

In [ ]:
# Identify all city sticker-related violations
sticker_mask = df['violation_description'].str.contains('CITY STICKER', case=False, na=False)
sticker_violations = df[sticker_mask]

# Show all unique codes and descriptions
code_summary = (
    sticker_violations
    .groupby(['violation_code', 'violation_description', 'fine_level1_amount'])
    .size()
    .reset_index(name='ticket_count')
    .sort_values('ticket_count', ascending=False)
)

print(f"City sticker tickets in sample: {len(sticker_violations):,}")
print(f"As % of all tickets: {len(sticker_violations)/len(df)*100:.1f}%\n")
code_summary

**Note on violation codes:** The primary codes are `0964125` (original), `0976170` (secondary), and `0964125B` (introduced after the 2012 fine restructuring for vehicles under 15,000 lbs). The code change created an administrative paper trail that helps us date when the new fine regime took effect.

---

## Part 2: The 2012 Fine Increase — Revenue Over Time

In 2012, the base fine rose from \$120 to \$200. The question is whether this increase was primarily a compliance measure or a revenue strategy. If it were about compliance, we'd expect ticket volumes to fall as residents adapted. If it were about revenue, volumes would remain stable or rise.

In [ ]:
# Flag all city sticker tickets
STICKER_CODES = ['0964125', '0976170', '0964125B']
df['is_sticker'] = df['violation_code'].isin(STICKER_CODES).astype(int)

# Monthly ticket volume
monthly = df.groupby('YearMonth')['is_sticker'].sum().reset_index()
monthly.columns = ['YearMonth', 'sticker_tickets']

# Mark the fine increase
fine_increase_date = pd.Timestamp('2012-07-01')

base = alt.Chart(monthly).mark_line(color='#c0392b').encode(
    x=alt.X('YearMonth:T', title='Month'),
    y=alt.Y('sticker_tickets:Q', title='City Sticker Tickets Issued'),
    tooltip=['YearMonth:T', 'sticker_tickets:Q']
).properties(
    title='Monthly City Sticker Ticket Volume (1% sample)',
    width=650,
    height=300
)

rule = alt.Chart(pd.DataFrame({'date': [fine_increase_date]})).mark_rule(
    color='black', strokeDash=[6, 3]
).encode(x='date:T')

label = alt.Chart(pd.DataFrame({
    'date': [fine_increase_date],
    'text': ['Fine increase: $120 → $200']
})).mark_text(
    align='left', dx=6, dy=-120, color='black', fontSize=11
).encode(x='date:T', text='text:N')

(base + rule + label)

---

## Part 3: The Revenue Calculation

Using the 1% sample, we can estimate the total revenue impact of the fine increase across the pre- and post-2012 periods.

In [ ]:
# Pre/post split on old codes (0964125 and 0976170 — pre-restructuring codes)
old_codes = df[df['violation_code'].isin(['0964125', '0976170'])]
n_old_code_tickets = len(old_codes)

fine_old = 120
fine_new = 200
fine_increase = fine_new - fine_old

# Revenue impact estimate (extrapolated from 1% sample)
revenue_increase_sample = n_old_code_tickets * fine_increase
revenue_increase_full = revenue_increase_sample * 100  # scale up from 1% sample

print(f"Tickets on old codes in sample: {n_old_code_tickets:,}")
print(f"Fine increase per ticket: ${fine_increase}")
print(f"Estimated additional revenue (1% sample): ${revenue_increase_sample:,.0f}")
print(f"Estimated additional revenue (full dataset): ${revenue_increase_full:,.0f}")
print(f"\nThat is roughly ${revenue_increase_full/1e6:.1f} million in additional revenue")
print("extracted from residents who couldn't afford or forgot to renew their city sticker.")

---

## Part 4: When Are These Tickets Issued?

Ticket timing reveals whether enforcement is opportunistic. If city sticker tickets cluster at month-end or fiscal year boundaries, that suggests quota-driven enforcement rather than public safety rationale.

In [ ]:
sticker_df = df[df['is_sticker'] == 1].copy()
sticker_df['month_of_year'] = sticker_df['issue_date'].dt.month
sticker_df['day_of_week'] = sticker_df['issue_date'].dt.day_name()

# By month of year
by_month = sticker_df.groupby('month_of_year').size().reset_index(name='count')
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
by_month['month_name'] = by_month['month_of_year'].map(month_names)

chart_month = alt.Chart(by_month).mark_bar(color='#2c3e50').encode(
    x=alt.X('month_name:N', sort=list(month_names.values()), title='Month'),
    y=alt.Y('count:Q', title='Tickets Issued'),
    tooltip=['month_name:N', 'count:Q']
).properties(
    title='City Sticker Tickets by Month of Year',
    width=600, height=280
)

# By day of week
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
by_day = sticker_df.groupby('day_of_week').size().reset_index(name='count')

chart_day = alt.Chart(by_day).mark_bar(color='#2980b9').encode(
    x=alt.X('day_of_week:N', sort=day_order, title='Day of Week'),
    y=alt.Y('count:Q', title='Tickets Issued'),
    tooltip=['day_of_week:N', 'count:Q']
).properties(
    title='City Sticker Tickets by Day of Week',
    width=600, height=280
)

(chart_month & chart_day)

---

## Part 5: Fine Escalation — The Debt Trap

Chicago doubles fines for unpaid tickets after a set period. For a resident who cannot pay a \$200 city sticker fine, this becomes \$400. Unpaid tickets trigger vehicle booting and impoundment — additional fees that can exceed the original fine many times over. This section examines how many sticker tickets went unpaid and what the escalated liability looks like.

In [ ]:
if 'ticket_queue' in sticker_df.columns:
    payment_status = sticker_df['ticket_queue'].value_counts().reset_index()
    payment_status.columns = ['status', 'count']
    payment_status['pct'] = (payment_status['count'] / payment_status['count'].sum() * 100).round(1)

    chart_status = alt.Chart(payment_status).mark_bar(color='#e67e22').encode(
        x=alt.X('count:Q', title='Number of Tickets'),
        y=alt.Y('status:N', sort='-x', title='Ticket Status'),
        tooltip=['status:N', 'count:Q', 'pct:Q']
    ).properties(
        title='City Sticker Ticket Outcomes',
        width=600, height=280
    )

    display(chart_status)

    unpaid = payment_status[payment_status['status'].str.contains('Delinquent|Bankruptcy|Dismissed', na=False)]['count'].sum()
    total = payment_status['count'].sum()
    print(f"Tickets not fully resolved: {unpaid:,} ({unpaid/total*100:.1f}%)")
else:
    print("Column 'ticket_queue' not found — check your dataset column names with df.columns")

---

## Summary: What the Data Shows

City sticker tickets are one of Chicago's most mechanically administered fines — issued at scale, with predictable seasonal and weekly patterns, and with a fine structure that has grown significantly steeper over time.

The 2012 fine increase from \$120 to \$200 generated tens of millions of dollars in additional city revenue. That revenue came disproportionately from residents living in lower-income neighbourhoods, for whom the annual sticker cost and the risk of an unpaid fine compounding into debt is most acute.

This notebook is part of a broader investigation into how municipal fine systems function as a form of regressive taxation. The data, methods, and findings are fully reproducible from the source dataset.

---

**Data source:** ProPublica Illinois  
**Full dataset:** [Chicago Data Portal](https://data.cityofchicago.org/)  
**Related reading:** [ProPublica — 'Driven Into Debt'](https://features.propublica.org/driven-into-debt/chicago-ticket-debt-bankruptcy/)